# Exploring the AI Entity Context endpoint

`GET /api/v1/{entityType}/name/{fqn}/context` (MCP tool `get_asset_context`) returns the full
**Context Profile** of an asset in one call: it walks the knowledge graph — schema, column-level
lineage, data profile, data quality, glossary, tags, and attached Context Center articles — and
renders it as Markdown (for an LLM) or JSON (for code). The response is **RBAC-filtered for the
caller**, so PII column profiles are masked unless you own the asset / are admin.

Validated live against a `banking-redshift` demo instance.

## Prerequisites
```bash
pip install requests
export AI_SDK_HOST="http://localhost:8585"
export AI_SDK_TOKEN="<your-jwt>"   # a user or admin personal access token
```

In [17]:
import os
from pathlib import Path

# Secrets live in a gitignored .env next to this notebook — never commit them.
# Copy .env.example to .env and fill in your values (one KEY=value per line).
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    print(f"Loaded secrets from {env_path.resolve()}")
else:
    print(".env not found — copy .env.example to .env and fill in your values.")

Loaded secrets from /Users/pmbrull/conductor/workspaces/ai-sdk/lagos/cookbook/persona-aware-context/.env


In [18]:
import os
import requests
from urllib.parse import quote

HOST = os.environ.get("AI_SDK_HOST", "http://localhost:8585").rstrip("/")
TOKEN = os.environ["AI_SDK_TOKEN"]  # read from env — never hard-code a token here
HEADERS = {"Authorization": f"Bearer {TOKEN}"}


def get_context(fqn, entity_type="tables", fmt="markdown", query=None):
    """AI Entity Context for one asset.

    fmt="markdown" -> str (OKF document);  fmt="json" -> dict (AIContext).
    query          -> optional natural-language question that biases the excerpt
                      of *truncated* attached knowledge toward the relevant passage.
    """
    url = f"{HOST}/api/v1/{entity_type}/name/{quote(fqn, safe='')}/context"
    params = {"format": fmt}
    if query:
        params["query"] = query
    resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json() if fmt == "json" else resp.text

## 1. The rendered Context Profile (Markdown)

This is what you drop straight into an LLM's context window.

In [19]:
from IPython.display import Markdown, display

TABLE = "banking-redshift.dev.marts_core.dim_customers"
display(Markdown(get_context(TABLE)))

---
type: "table"
title: "banking-redshift.dev.marts_core.dim_customers"
description: "Customer dimension enriched with rollups across deposits, loans and wealth holdings, plus a derived value_segment used to drive downstream banking and…"
fullyQualifiedName: "banking-redshift.dev.marts_core.dim_customers"
tags: ["PII.Sensitive"]
timestamp: "2026-07-22T14:52:29.271Z"
---

Customer dimension enriched with rollups across deposits, loans and wealth holdings, plus a derived value_segment used to drive downstream banking and marketing analytics. Carries PII (SSN, DOB, email, phone, mailing address) — restrict access per the bank's privacy policy. Grain: one row per customer_id.

# Schema

| Column | Type | Constraint | Description |
|--------|------|------------|-------------|
| customer_id | character varying(30) | NULL | Stable globally-unique customer identifier — primary key. |
| first_name | character varying(120) | NULL | Customer legal first name. |
| last_name | character varying(80) | NULL | Customer legal last name. |
| email | character varying(240) | NULL | Lowercased customer email address. PII. |
| phone | character varying(20) | NULL | Customer phone number. PII. |
| ssn | character varying(11) | NULL | US Social Security Number. PII — restricted to retail customers. |
| date_of_birth | date | NULL | Customer date of birth. PII. |
| customer_segment | character varying(30) | NULL | Marketing/risk segment code. |
| customer_type | character varying(45) | NULL | Customer type — retail, business, or private_banking. |
| branch_id | character varying(30) | NULL | Home branch of the customer (FK to dim_branches). |
| primary_employee_id | character varying(30) | NULL | Personal banker assigned to the customer. |
| kyc_status | character varying(30) | NULL | Current KYC verification status. |
| risk_band | character varying(30) | NULL | Bank-assigned risk band. |
| created_at | timestamp without time zone | NULL | Timestamp the customer record was created. |
| updated_at | timestamp without time zone | NULL | Timestamp the customer record was last updated. |
| mailing_address_line_1 | character varying(120) | NULL | Primary mailing street address line. PII. |
| mailing_address_line_2 | character varying(120) | NULL | Secondary mailing address line. PII. |
| mailing_city | character varying(80) | NULL | Mailing city. PII. |
| mailing_state | character varying(20) | NULL | Mailing state / province code. PII. |
| mailing_postal_code | character varying(20) | NULL | Mailing ZIP / postal code. PII. |
| mailing_country | character varying(20) | NULL | Mailing country. |
| primary_email | character varying(120) | NULL | Primary verified email contact (from customer_contacts). PII. |
| primary_mobile | character varying(120) | NULL | Primary verified mobile contact (from customer_contacts). PII. |
| last_kyc_date | date | NULL | Date of the customer's most recent KYC check. |
| last_kyc_status | character varying(30) | NULL | Status of the most recent KYC check. |
| current_fico_score | integer | NULL | Most recent FICO bureau score value (300-850). |
| current_fico_score_date | date | NULL | Date of the most recent FICO score pull. |
| current_fico_bureau | character varying(45) | NULL | Bureau that produced the most recent FICO score. |
| total_deposit_balance | numeric(38,4) | NULL | Sum of positive balances across the customer's deposit-category accounts (zero when the customer has no deposit accounts), in USD. |
| account_count | bigint | NULL | Number of accounts the customer is the primary holder on. |
| total_loan_balance | numeric(38,4) | NULL | Sum of outstanding balances across active loans (excludes paid_off and charged_off loans), in USD. |
| loan_count | bigint | NULL | Number of active loans (excludes paid_off and charged_off). |
| aum | numeric(38,0) | NULL | Assets under management; sum of market_value across the customer's wealth holdings (zero when the customer has no holdings), in USD. |
| value_segment | character varying(15) | NULL | Derived bucket used for segmentation: private_banking (aum >= 250k), high_value (deposits >= 50k), mass_market (deposits >= 5k), emerging (otherwise). |

# Data Model

**Type:** `DBT` · **Path:** `models/marts/core/dim_customers.sql` · **Project:** `banking_redshift`

```sql
-- Customer dimension enriched with rollup totals across deposits, loans and
-- wealth holdings, plus a value_segment bucket used by downstream marts.
-- Grain: one row per customer_id.

with customers as (
    select * from "dev"."intermediate"."int_customers__360"
),

accounts as (
    select * from "dev"."intermediate"."int_accounts__enriched"
),

loans as (
    select * from "dev"."intermediate"."int_loans__delinquency"
),

holdings as (
    select * from "dev"."intermediate"."int_holdings__valued"
),

deposit_agg as (
    select
        customer_id,
        sum(
            case
                when product_category = 'deposit' and balance > 0
                    then balance
                else 0
            end
        ) as total_deposit_balance,
        count(*) as account_count
    from accounts
    group by customer_id
),

loan_agg as (
    select
        customer_id,
        sum(
            case
                when status not in ('paid_off', 'charged_off') then balance
                else 0
            end
        ) as total_loan_balance,
        sum(
            case
                when status not in ('paid_off', 'charged_off') then 1
                else 0
            end
        ) as loan_count
    from loans
    group by customer_id
),

holdings_agg as (
    select
        customer_id,
        sum(market_value) as aum
    from holdings
    where customer_id is not null
    group by customer_id
),

joined as (
    select
        c.customer_id,
        c.first_name,
        c.last_name,
        c.email,
        c.phone,
        c.ssn,
        c.date_of_birth,
        c.customer_segment,
        c.customer_type,
        c.branch_id,
        c.primary_employee_id,
        c.kyc_status,
        c.risk_band,
        c.created_at,
        c.updated_at,
        c.mailing_address_line_1,
        c.mailing_address_line_2,
        c.mailing_city,
        c.mailing_state,
        c.mailing_postal_code,
        c.mailing_country,
        c.primary_email,
        c.primary_mobile,
        c.last_kyc_date,
        c.last_kyc_status,
        c.current_fico_score,
        c.current_fico_score_date,
        c.current_fico_bureau,
        coalesce(d.total_deposit_balance, 0) as total_deposit_balance,
        coalesce(d.account_count, 0)         as account_count,
        coalesce(l.total_loan_balance, 0)    as total_loan_balance,
        coalesce(l.loan_count, 0)            as loan_count,
        coalesce(h.aum, 0)                   as aum
    from customers c
    left join deposit_agg d  on c.customer_id = d.customer_id
    left join loan_agg l     on c.customer_id = l.customer_id
    left join holdings_agg h on c.customer_id = h.customer_id
)

select
    *,
    case
        when aum >= 250000                   then 'private_banking'
        when total_deposit_balance >= 50000  then 'high_value'
        when total_deposit_balance >= 5000   then 'mass_market'
        else 'emerging'
    end as value_segment
from joined
```

# Business Definitions

### Account
`BankingCore.Account`

A financial account held by one or more customers at the bank.

### Customer
`BankingCore.Customer`

A person or legal entity with a banking relationship.

### Private Banking Customer
`BankingCore.Customer.PrivateBankingCustomer`

A high-net-worth customer served by the private banking and wealth desk.

### Data Subject
`DataPrivacyPII.DataSubject`

An identified or identifiable natural person whose personal data is processed. Under CCPA/CPRA called a 'consumer' (Cal. Civ. Code §1798.140(g)).

**Regulatory citation.** GDPR Article 4(1).

**Rights.** Subjects have rights to access (Article 15), rectification (Article 16), erasure (Article 17, 'right to be forgotten'), restriction (Article 18), portability (Article 20), and to object (Article 21).

### Beneficial Owner
`DataPrivacyPII.PII.BeneficialOwner`

An individual who ultimately owns or controls a legal entity (>=25% threshold per FinCEN).

### Branch
`BankingCore.Branch`

A physical retail location where customers transact in person.

### Credit Card Account
`BankingCore.Account.CreditCardAccount`

A revolving credit account linked to one or more cards.

# Knowledge Articles

### Customer-360 Data Model
`customer-360-data-model`

# Customer 360 Data Model

## Access control and PII retention policy
This section governs personally identifiable information and access control for customer data. The dim_customers table stores restricted PII: ssn, tax_id, date_of_birth, email, phone and mailing address. Every non-owner and non-admin caller receives these sensitive columns masked at the server, so notebooks, dashboards and automated agents only ever receive masked values unless the caller is a registered owner of the asset. To read a raw ssn or tax_id an analyst must first be granted the pii-viewer role through the access-governance approval workflow, and…

Sections: Customer 360 Data Model · Access control and PII retention policy · Refresh schedule freshness and pipeline SLA · Known data quality issues and test coverage

_Excerpt — fetch the full content with get_knowledge_content(entityType=`page`, fqn=`customer-360-data-model`)._

### dbt Model Layers (banking-redshift)
`dbt-model-layers`

# dbt Model Layers

The banking-redshift dbt project follows the standard staging /
intermediate / marts pattern.

```
seeds (raw_*.csv)
  └── staging  (dbt_staging)             # 1:1 with the raw layer
        └── intermediate (dbt_intermediate)  # business logic, no joins
              └── marts (dbt_marts_*)         # consumer-facing facts/dims
```

## Marts by domain

| Domain     | Schema              | Highlights                                              |
|------------|---------------------|---------------------------------------------------------|
| Core       | `dbt_marts_core`    | `dim_customers`, `fct_transactions`, `fct_card_*`       |
| Finance    | `dbt_marts_finance` | `fct_daily_balances`, `fct_monthly_pnl`, `fct_nim`      |
| Risk       | `dbt_marts_risk`    | `fct_loan_loss_provision`, `fct_aml_pipeline`           |
| Wealth     | `dbt_marts_wealth`  | `fct_aum`, `fct_trades`, `dim_holdings`                 |
| Marketing  | `dbt_marts_marketing`| `fct_campaign_attribution`, `fct_customer_engagement`  |

Lineage to the upstream Redshift raw layer is captured by the dbt ingestion
workflow; lineage from the marts to Superset dashboards is captured by the
Superset ingestion workflow.

# Lineage

**Upstream:**
- `banking-redshift.dev.intermediate.int_holdings__valued`
  - `market_value → value_segment`
  - `market_value → aum`
- `banking-redshift.dev.intermediate.int_accounts__enriched`
  - `branch_id → branch_id`
  - `created_at → created_at`
  - `customer_id → customer_id`
  - `updated_at → updated_at`
- `banking-redshift.dev.intermediate.int_customers__360`
  - `current_fico_bureau → current_fico_bureau`
  - `current_fico_score_date → current_fico_score_date`
  - `customer_id → customer_id`
  - `kyc_status → kyc_status`
  - `risk_band → risk_band`
  - `mailing_address_line_1 → mailing_address_line_1`
  - `mailing_address_line_2 → mailing_address_line_2`
  - `last_kyc_status → last_kyc_status`
  - `mailing_postal_code → mailing_postal_code`
  - `mailing_city → mailing_city`
  - `phone → phone`
  - `customer_segment → customer_segment`
  - `customer_type → customer_type`
  - `primary_employee_id → primary_employee_id`
  - `updated_at → updated_at`
  - `date_of_birth → date_of_birth`
  - `last_kyc_date → last_kyc_date`
  - `last_name → last_name`
  - `current_fico_score → current_fico_score`
  - `mailing_country → mailing_country`
  - `ssn → ssn`
  - `mailing_state → mailing_state`
  - `created_at → created_at`
  - `primary_email → primary_email`
  - `branch_id → branch_id`
- `banking-redshift.dev.intermediate.int_loans__delinquency`
  - `status → loan_count`
  - `status → total_loan_balance`
  - `balance → total_loan_balance`

**Downstream:**
- `banking-superset.model.10`
  - `customer_id → 76`
  - `first_name → 77`
  - `last_name → 78`
  - `email → 79`
  - `phone → 80`
  - `ssn → 81`
  - `date_of_birth → 82`
  - `customer_segment → 83`
  - `customer_type → 84`
  - `branch_id → 85`
  - `primary_employee_id → 86`
  - `kyc_status → 87`
  - `risk_band → 88`
  - `created_at → 89`
  - `updated_at → 90`
  - `mailing_address_line_1 → 91`
  - `mailing_address_line_2 → 92`
  - `mailing_city → 93`
  - `mailing_state → 94`
  - `mailing_postal_code → 95`
  - `mailing_country → 96`
  - `primary_email → 97`
  - `primary_mobile → 98`
  - `last_kyc_date → 99`
  - `last_kyc_status → 100`


_Column mappings are capped at 25 per edge — fetch the full lineage graph with get_entity_lineage(entityType=`table`, fqn=`banking-redshift.dev.marts_core.dim_customers`)._

# Data Profile

**Row count:** 5001

| Column | Null % | Distinct | Min | Max |
|--------|--------|----------|-----|-----|
| customer_id | 0% | 2496 | 11 | 11 |
| first_name | 0% | 795 | 2 | 32 |
| last_name | 12% | 722 | 2 | 11 |
| email | 0% | 2495 | 12 | 36 |
| phone | 0% | 2515 | 14 | 14 |
| ssn | 11% | 2292 | 11 | 11 |
| date_of_birth | 0% | 2273 | 1936-10-06 | 2050-01-01 |
| customer_segment | 0% | 8 | 4 | 15 |
| customer_type | 0% | 3 | 6 | 15 |
| branch_id | 0% | 75 | 7 | 7 |
| primary_employee_id | 0% | 50 | 9 | 9 |
| kyc_status | 0% | 4 | 6 | 7 |
| risk_band | 0% | 4 | 3 | 8 |
| created_at | 0% | 2542 | 2015-01-01T10:47:36 | 2026-05-22T16:12:13 |
| updated_at | 0% | 1 | 2026-05-22T12:00:00 | 2026-05-22T12:00:00 |
| mailing_address_line_1 | 57% | 1039 | 12 | 35 |
| mailing_address_line_2 | 91% | 210 | 8 | 9 |
| mailing_city | 57% | 1050 | 6 | 22 |
| mailing_state | 55% | 59 | 2 | 2 |
| mailing_postal_code | 57% | 1093 | 5 | 5 |
| mailing_country | 58% | 1 | 3 | 3 |
| primary_email | 0% | 2464 | 12 | 34 |
| primary_mobile | 100% | 0 |  |  |
| last_kyc_date | 27% | 1179 | 2015-07-20 | 2026-05-22 |
| last_kyc_status | 27% | 4 | 6 | 7 |
| current_fico_score | 30% | 364 | 450 | 820 |
| current_fico_score_date | 30% | 894 | 2023-01-08 | 2026-05-22 |
| current_fico_bureau | 30% | 3 | 7 | 10 |
| total_deposit_balance | 0% | 1784 | 0.0 | 2038046.41 |
| account_count | 0% | 7 | 0 | 6 |
| total_loan_balance | 0% | 1014 | 0.0 | 1515655.2 |
| loan_count | 0% | 5 | 0 | 4 |
| aum | 0% | 112 | 0.0 | 51799993 |
| value_segment | 0% | 4 | 8 | 15 |

# Data Quality

Tests — passed: 0, failed: 0, aborted: 0


## 2. The same profile as JSON

Every field of the `AIContext` object, for programmatic use.

In [20]:
ctx = get_context(TABLE, fmt="json")

print("Top-level fields:", ", ".join(ctx))
print()
print("description   :", ctx["description"])
print("upstream      :", ctx["upstream"])
print("downstream    :", ctx["downstream"])
print("tags          :", ctx.get("tags"))
print("glossaryTerms :", [g.get("name") for g in (ctx.get("glossaryTerms") or [])])
print("articles      :", [a.get("name") for a in (ctx.get("articles") or [])])

Top-level fields: id, fullyQualifiedName, entityType, description, tags, glossaryTerms, metrics, articles, upstream, downstream, upstreamEdges, downstreamEdges, assetContext, observability, generatedAt

description   : Customer dimension enriched with rollups across deposits, loans and wealth holdings, plus a derived value_segment used to drive downstream banking and marketing analytics. Carries PII (SSN, DOB, email, phone, mailing address) — restrict access per the bank's privacy policy. Grain: one row per customer_id.

upstream      : ['banking-redshift.dev.intermediate.int_holdings__valued', 'banking-redshift.dev.intermediate.int_accounts__enriched', 'banking-redshift.dev.intermediate.int_customers__360', 'banking-redshift.dev.intermediate.int_loans__delinquency']
downstream    : ['banking-superset.model.10']
tags          : ['PII.Sensitive', 'Tier.Tier1']
glossaryTerms : ['Account', 'Customer', 'PrivateBankingCustomer', 'DataSubject', 'BeneficialOwner', 'Branch', 'CreditCardAccount

## 3. Traverse the graph: column-level lineage

The profile names its neighbours — with column mappings — so an agent can hop to the next node's
`/context` and keep walking.

In [12]:
for edge in ctx["upstreamEdges"]:
    print("UP  ", edge["fullyQualifiedName"])
    for m in (edge.get("columns") or [])[:3]:
        print("       ", ", ".join(m["fromColumns"]), "->", m["toColumn"])

UP   banking-redshift.dev.intermediate.int_holdings__valued
        banking-redshift.dev.intermediate.int_holdings__valued.market_value -> banking-redshift.dev.marts_core.dim_customers.value_segment
        banking-redshift.dev.intermediate.int_holdings__valued.market_value -> banking-redshift.dev.marts_core.dim_customers.aum
UP   banking-redshift.dev.intermediate.int_accounts__enriched
        banking-redshift.dev.intermediate.int_accounts__enriched.branch_id -> banking-redshift.dev.marts_core.dim_customers.branch_id
        banking-redshift.dev.intermediate.int_accounts__enriched.created_at -> banking-redshift.dev.marts_core.dim_customers.created_at
        banking-redshift.dev.intermediate.int_accounts__enriched.customer_id -> banking-redshift.dev.marts_core.dim_customers.customer_id
UP   banking-redshift.dev.intermediate.int_customers__360
        banking-redshift.dev.intermediate.int_customers__360.current_fico_bureau -> banking-redshift.dev.marts_core.dim_customers.current_fico_b

## 4. Schema + data profile

`assetContext.table` carries the structural view; `observability` carries the profile (RBAC-masked).

In [13]:
tbl = ctx["assetContext"]["table"]
print("columns      :", len(tbl["columns"]))
print("primaryKey   :", tbl.get("primaryKey"))
print("frequentJoins:", tbl.get("frequentJoins"))
print()

obs = ctx["observability"]
print(f"rowCount: {obs['rowCount']:.0f}")
for cp in obs["columnProfiles"][:8]:
    print(f"  {cp['name']:22} null={cp['nullProportion']:.2f}  distinct={cp['distinctCount']}")

columns      : 34
primaryKey   : []
frequentJoins: []

rowCount: 5001
  customer_id            null=0.00  distinct=2496.0
  first_name             null=0.00  distinct=795.0
  last_name              null=0.12  distinct=722.0
  email                  null=0.00  distinct=2495.0
  phone                  null=0.00  distinct=2515.0
  ssn                    null=0.11  distinct=2292.0
  date_of_birth          null=0.00  distinct=2273.0
  customer_segment       null=0.00  distinct=8.0


## 5. Question-aware context — the `query` param

`query` does **not** filter or change *which* asset you get. When an attached knowledge item (a
glossary definition or a Context Center article) is too long to inline in full, the excerpt shown is
picked by **semantic similarity to `query`** — the most relevant chunk of that item — instead of the
positional lead. It gives the agent a decision-grade snippet of a long doc.

It only *visibly* changes the output when **all** of these hold (server constants):

- the item is **truncated** — body `> ~1500 chars` (`MAX_ITEM_CHARS`), so `contentTruncated=True`;
- it spans **multiple chunks** — the body is chunked every **~380 words** (`MAX_WORDS_PER_CHUNK`), so
  each *topic* needs its own ~400-word section to land in a different chunk;
- **vector embeddings are enabled** on the instance (otherwise it falls back to a structural preview).

Below, the `customer-360-data-model` article on `dim_customers` has three ~400-word sections (PII,
freshness, data quality). The **same asset** returns a **different passage** per question.

In [14]:
def article_excerpt(fqn, article_name, query=None):
    """Return the (possibly query-biased) excerpt of one attached article."""
    ctx = get_context(fqn, fmt="json", query=query)
    for a in (ctx.get("articles") or []):
        if a.get("name") == article_name:
            content = a.get("content") or ""
            # a query excerpt carries a "name: ...; | <body>" chunk header — keep the body
            body = content.split(" | ", 1)[-1].strip()
            return a.get("contentTruncated"), body
    return None, ""


ARTICLE = "customer-360-data-model"
QUESTIONS = [
    None,
    "what is the PII access control and ssn retention policy?",
    "when does the table refresh and what is the freshness SLA?",
    "what are the known data quality issues and dbt tests?",
]

for q in QUESTIONS:
    truncated, body = article_excerpt(TABLE, ARTICLE, query=q)
    label = "NO QUERY (positional lead)" if q is None else f"query = {q!r}"
    print("=" * 90)
    print(f"{label}   (truncated={truncated})")
    print("-" * 90)
    print(body[:300], "...\n")

NO QUERY (positional lead)   (truncated=True)
------------------------------------------------------------------------------------------
# Customer 360 Data Model

## Access control and PII retention policy
This section governs personally identifiable information and access control for customer data. The dim_customers table stores restricted PII: ssn, tax_id, date_of_birth, email, phone and mailing address. Every non-owner and non-ad ...

query = 'what is the PII access control and ssn retention policy?'   (truncated=True)
------------------------------------------------------------------------------------------
title: Customer-360 Data Model; description: # Customer 360 Data Model ## Access control and PII retention policy This section governs personally identifiable information and access control for customer data. The dim_customers table stores restricted PII: ssn, tax_id, date_of_birth, email, phone and ...

query = 'when does the table refresh and what is the freshness SLA?'   (tr